# DentaBridge: tuning Qwen3-VL-4B on one free T4

Settings: **Accelerator** = GPU T4, **Internet** = On. Add the private dataset `dentabridge-code`
(built by `make kaggle-bundle`) as an input. Set `STAGES` below: the default only installs, loads the
data and runs ten training steps, and prints how long the full run would take. Then add
`train`, `merge`, `eval` and `zeroshot` and run again (a session lasts at most 12 hours; the weekly
GPU quota is 30 hours). The same stages run on Colab with
`WORK=/content SCRATCH=/content/scratch ROOT=/content/dentabridge`.

In [ ]:
STAGES = ["setup", "data", "smoke"]  # + "train", "merge", "eval", "zeroshot" for the full run

In [ ]:
import pathlib
import shutil
import subprocess

ROOT = pathlib.Path("/kaggle/working/dentabridge")
if not ROOT.exists():
    inputs = pathlib.Path("/kaggle/input")
    archive = next(inputs.rglob("dentabridge-code.tar.gz"), None)
    if archive:
        subprocess.run(["tar", "-xzf", str(archive), "-C", "/kaggle/working"], check=True)
    else:  # Kaggle unpacks an uploaded archive into a folder
        folder = next(p.parent for p in inputs.rglob("COMMIT") if p.parent.name == "dentabridge")
        shutil.copytree(folder, ROOT, ignore=shutil.ignore_patterns("._*"))
print("code at commit", (ROOT / "COMMIT").read_text().strip())


def run_stage(name):
    if name not in STAGES:
        print("skipped:", name)
        return
    process = subprocess.Popen(
        ["bash", str(ROOT / "scripts/kaggle/run.sh"), name],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError(f"stage {name} failed with exit code {process.returncode}")

## 1. Install

In [ ]:
run_stage("setup")

## 2. Data (AKU and AlphaDent from Zenodo, CC BY 4.0)

In [ ]:
run_stage("data")

## 3. Smoke test: ten steps, and the estimate for the full run

In [ ]:
run_stage("smoke")

## 4. Train (resumes from the newest checkpoint if the session was interrupted)

In [ ]:
run_stage("train")

## 5. Merge the adapter and score the validation split

In [ ]:
run_stage("merge")

In [ ]:
run_stage("eval")

## 6. Baseline: the same prompt on the untuned model

In [ ]:
run_stage("zeroshot")

## 7. Results

In [ ]:
import glob
import pathlib

for path in sorted(glob.glob("/kaggle/working/out/*/val/val.md")):
    print(pathlib.Path(path).read_text())

Download from the **Output** tab: `out/lean-t4/adapter`, every `run.json` and `out/*/val/`.
The `checkpoint-*` folders are only for resuming.